In [1]:
import pandas as pd
# from langchain_gigachat import GigaChat
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, List, Tuple, Union, Literal, Dict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd
import re
import subprocess
from tabulate import tabulate

from tqdm import tqdm
import json
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [2]:
from dotenv import load_dotenv
load_dotenv()

gigatoken = os.getenv('credentials')
chatgpt_token = os.getenv('OPENAI_API_KEY')


In [3]:
dataset = pd.read_csv('scenarios_with_funcs.csv')
dataset.index.name = "Index"
dataset = dataset[dataset['target'] != -1]
dataset

,scene_name,entry_condition,user_message,category,target,scene_text,funcs
Index,,,,,,,
0,Устранение проблем с блокировкой карты,Запрос пользователя связан с одной из перечисл...,"Я не могу использовать свою карту, она, похоже...",Проблемы с картами,1,Идентификация причины блокировки карты. Провер...,"[""check_transaction_history(), confirm_custome..."
1,Подтверждение транзакции,Запрос пользователя связан с одной из перечисл...,Я не вижу на своем счете транзакцию за 5000 ру...,Транзакции,1,Проверка данных транзакции. Верификация источн...,"[""verify_fund_source(), check_transaction_stat..."
2,Проверка статуса страхового полиса,Запрос пользователя связан с одной из перечисл...,Какой статус моего страхового полиса?,Страхование,1,Проверка идентификационной информации клиента....,"[""check_policy_status(), suggest_policy_terms(..."
3,Консультация по инвестиционным стратегиям,Запрос пользователя связан с одной из перечисл...,Какой у вас курс валюты на сегодня?,Инвестиции,0,Анализ финансовых целей клиента. Оценка рисков...,"[""suggest_investment_options()""]"
4,Оформление кредита,Запрос пользователя связан с одной из перечисл...,"Может, расскажете про кредиты? Интересно, что ...",Кредитование,1,Проверка кредитоспособности клиента. Сбор необ...,"[""collect_documents(), suggest_credit_products..."
...,...,...,...,...,...,...,...
190,Проверка активности счета,Запрос пользователя связан с одной из перечисл...,У меня на счету появилось какое-то подозритель...,Безопасность счета,1,Идентификация клиента. Запрос информации о пос...,"[""fetch_recent_transactions()""]"
191,Блокировка банковской карты при утере,Запрос пользователя связан с одной из перечисл...,"Слушай, кажется, я оставил карту в другом горо...",Проблемы с картами,1,Получение информации о состоянии карты. Провер...,"[""verify_client_data(), locate_last_activity()..."
192,Проверка статуса транзакции,Запрос пользователя связан с одной из перечисл...,"Я хочу узнать, прошла ли моя последняя транзак...",Транзакции,1,Получение информации о транзакции. Проверка ст...,"[""check_transaction_status()"", ""analyze_delay_..."


In [4]:
dataset.loc[77]['scene_text'],dataset.loc[77]['user_message'], dataset.loc[77]['funcs']

('Проверка данных клиента. Анализ текущего кредитного состояния. - Используй функцию analyze_credit_status(). Оценка доходов клиента. - Включи информацию с помощью функции evaluate_income(). Определение нового лимита на основе анализа. - Предложи варианты лимита на основе анализа. Оформление увеличения лимита. - Используй функцию process_credit_limit_increase().',
 'Почему у вас такие высокие проценты по кредитам?',
 '["analyze_credit_status(), evaluate_income(), process_credit_limit_increase()"]')

In [5]:
import json
import pandas as pd

def decode_unicode_column(dataset, column_name='funcs'):
    """Декодирует значения в указанной колонке, если они в формате Unicode."""
    decoded_funcs = []
    for funcs in dataset[column_name]:
        if isinstance(funcs, str):
            try:
                decoded = json.loads(funcs)
                decoded_funcs.append(decoded)
            except json.JSONDecodeError:

                decoded_funcs.append(funcs)
        else:
            decoded_funcs.append(funcs)
    dataset[column_name] = decoded_funcs

decode_unicode_column(dataset, 'funcs')
dataset['funcs'].head()


Index
0    [check_transaction_history(), confirm_customer...
1    [verify_fund_source(), check_transaction_statu...
2    [check_policy_status(), suggest_policy_terms()...
3                       [suggest_investment_options()]
4     [collect_documents(), suggest_credit_products()]
Name: funcs, dtype: object

In [6]:
prompt_template = """ Ты реврайтер сообщения пользователя. Твоя задача взять старое сообщение пользователя {old_user_message} 
и подогнать под первый пункт сценария из  {scene_text} в котором встречается любая функция из списка {funcs}, тобишь вернуть 'new_user_message'. 
ТО ЕСТЬ ТВОЯ ОСНОВАНАЯ ЗАДАЧА - написать ДВА сообщения, разделённые ";" , первое из них - то на которое должны вывзваться функция из списка функций. Второе - подтверждающее сообщение для потенциального вопроса от системы.
Пример 1 : 
OLD_USER_MESSAGE = Я просто не знаю, как с кредитами обращаться.'
SCENE_TEXT ='Проверка данных клиента. Анализ текущих условий кредита. - Используй функцию analyze_current_terms(). Предложение новых условий. - Используй функцию suggest_new_terms()
FUNCS = analyze_current_terms(), suggest_new_terms()
ТВОЙ ответ: 
я не помню что у меня с кредитами, хочу узнать свои условия кредита; мой id = 1234
ПРИМЕР 2: 
OLD_USER_MESSAGE = Не могли бы вы проверить статус моей транзакции на 5000 рублей и подтвердить, что она была успешно выполнена?
SCENE_TEXT =Идентификация причины блокировки карты. Проверка истории транзакций. - Используй функцию check_transaction_history(). Предложение решения проблемы. - Обсуди варианты разблокировки или получения новой карты.
FUNCS = check_transaction_history(), confirm_customer_actions()
ТВОЙ ответ: 
Не могли бы вы проверить статус моей транзакции на 5000 рублей и подтвердить, что она была успешно выполнена?; айди транзакции - 2281337

ТО ЕСТЬ,  надо сделать такой ответ, чтобы на него вызвалась функция. НЕ ПРОСИ ЯВНО ВЫЗЫВАТЬ НАЗВАНИЯ ФУНКИЙ - ТЫ О НИХ НЕ ЗНАЕШЬ!!!!!! ТВОЕ СООБЩЕНИЕ ДОЛЖНО БЫТЬ МАКСИМАЛЬНО ЧЕЛОВЕКОПОДОБНЫМ

ОТВЕТ ВОЗВРАЩАЙ ПРОСТО СТРОКОЙ - НОВОЕ СООБЩЕНИЕ, БЕЗ ВСЯКИХ ДОПОЛНИТЕЛНЬЫХ СЛОВ.
"""

system_prompt = ChatPromptTemplate.from_template(prompt_template)
generator = system_prompt | ChatOpenAI(
    model="gpt-4o", temperature=0.9, api_key=chatgpt_token
)

In [7]:
new_ums =[]
for idx,row in tqdm(dataset.iterrows()):
        res = generator.invoke({"old_user_message":row['user_message'],"scene_text":row['scene_text'],"funcs":row['funcs']})
        new_ums.append([msg for msg in res.content.split(";")])
        # print(f"старое сообщение -\n{row['user_message']} \nновое сообщение - \n{res.content}, \nСценарий {row['scene_text']}")
dataset['um_for_funcs'] = new_ums

195it [03:51,  1.19s/it]


In [8]:
dataset.head()
dataset.to_csv('scenarios_with_funcs_v2.csv',index=False)

In [9]:

data = pd.read_csv('scenarios_with_funcs_v2.csv')
data.head()

,scene_name,entry_condition,user_message,category,target,scene_text,funcs,um_for_funcs
0,Устранение проблем с блокировкой карты,Запрос пользователя связан с одной из перечисл...,"Я не могу использовать свою карту, она, похоже...",Проблемы с картами,1,Идентификация причины блокировки карты. Провер...,"['check_transaction_history(), confirm_custome...","['Я не могу использовать свою карту, могли бы ..."
1,Подтверждение транзакции,Запрос пользователя связан с одной из перечисл...,Я не вижу на своем счете транзакцию за 5000 ру...,Транзакции,1,Проверка данных транзакции. Верификация источн...,"['verify_fund_source(), check_transaction_stat...",['Я не вижу на своем счете транзакцию на 5000 ...
2,Проверка статуса страхового полиса,Запрос пользователя связан с одной из перечисл...,Какой статус моего страхового полиса?,Страхование,1,Проверка идентификационной информации клиента....,"['check_policy_status(), suggest_policy_terms(...",['Какой статус моего текущего страхового полис...
3,Консультация по инвестиционным стратегиям,Запрос пользователя связан с одной из перечисл...,Какой у вас курс валюты на сегодня?,Инвестиции,0,Анализ финансовых целей клиента. Оценка рисков...,['suggest_investment_options()'],"['Я хотел бы узнать, какие инвестиционные инст..."
4,Оформление кредита,Запрос пользователя связан с одной из перечисл...,"Может, расскажете про кредиты? Интересно, что ...",Кредитование,1,Проверка кредитоспособности клиента. Сбор необ...,"['collect_documents(), suggest_credit_products...","['Можете подсказать, какие документы нужно соб..."
